In [1]:
"""GPR Campaign Ploemeur — 06-06-2016.  Equivalent to seq06.m."""
from pathlib import Path
import sys
import importlib
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from gdp.data_io import load_mala
from gdp.preprocessing.filtering import filter_data, remove_mean
from gdp.preprocessing.gain import linear_gain
from gdp.preprocessing.image_processing import remove_svd
from gdp.preprocessing.trace_ops import align_traces

sys.path.insert(0, str(Path.cwd()))
import helper_functions.KirchhoffPylopsZeroOffset as KirchhoffPylopsZeroOffset
importlib.reload(KirchhoffPylopsZeroOffset)
from pylops.utils.wavelets import ricker
from helper_functions.migration import gazdag_migration, write_backprop_files

# Loading in data from 6 June 2016

In [2]:
DATA = Path.cwd() / 'fielddata' / 'raw_data' / '060616'
OUT_DIR = Path.cwd() / 'fielddata' / 'output'

# Process Reference

In [3]:
runs = list(range(0, 6)) + list(range(7, 39))

# MATLAB: R = D(:,:,1) — first profile is reference
ref_run = runs[0]
data_runs = runs[1:]

_prof_name = lambda n: 'prof' if n == 0 else f'prof{n}'
ref_raw, info = load_mala(str(DATA / _prof_name(ref_run)), return_object=False)
sf = info['frequency (GHz)']
n_traces = ref_raw.shape[1]
samples = ref_raw.shape[0]

# Preprocess reference
ref_bp = filter_data(ref_raw, fq=(0.02, 0.2), sfreq=sf, btype='bandpass')
ref_dc, _ = remove_mean(ref_bp, 299, 517)

ref_aligned, _, _ = align_traces(ref_dc, ref_dc, upsample=5, normalize=False, align_reference=True)

ref_svd, _ = remove_svd(ref_aligned, low_s=0, high_s=1)

t = np.arange(1, samples + 1) / sf
v = 0.10
dL = 0.05
max_d = 85.0
sc = 1.2
rad_cut = 300
depth = np.linspace(max_d, max_d - n_traces * dL, n_traces)
radius = np.linspace(0, v * 450 / 2, samples)

OUT_DIR.mkdir(exist_ok=True)
(OUT_DIR / 'processed').mkdir(exist_ok=True)

# Migration setup
t_mig  = t[:rad_cut]
f0_mig = 0.1                                        # centre frequency [GHz] — adjust to antenna

# Ricker wavelet (matches PylopsKirchoffMigration convention)
_period = 1.0 / f0_mig
_n_wav  = int(np.ceil(6 * _period / (t_mig[1] - t_mig[0])))
if _n_wav % 2 == 0:
    _n_wav += 1
wav_mig, _, wcenter_mig = ricker(t_mig[:_n_wav], f0=f0_mig)

# Image axes: x = radial distance from borehole [m], z spans borehole depth range
x_img = np.linspace(0, v * t_mig[-1] / 2, rad_cut)

(OUT_DIR / 'migrated').mkdir(exist_ok=True)
# Back-propagation gprMax grid -- lambda/20 at f0_mig (~20 cells/wavelength)
bp_dx       = v / (f0_mig * 20)                      # grid spacing [m]
bp_pml      = 10                                      # PML cells
bp_src_y    = (bp_pml + 1) * bp_dx                   # source just inside PML on borehole side [m]
bp_domain_y = float(x_img[-1]) + 2 * bp_pml * bp_dx  # radial extent + PML padding both sides [m]

# Process other profiles

In [ ]:
for run in data_runs:
    data, _ = load_mala(str(DATA / _prof_name(run)), return_object=False)
    n = min(data.shape[1], n_traces)

    d_bp = filter_data(data[:, :n], fq=(0.02, 0.2), sfreq=sf, btype='bandpass')
    d_dc, _ = remove_mean(d_bp, 299, 517)

    d_aligned, _, _ = align_traces(d_dc, ref_aligned[:, :n], upsample=5, normalize=True, align_reference=False)

    d_svd, _ = remove_svd(d_aligned, low_s=0, high_s=1)
    d_diff = d_svd - ref_svd[:, :n]
    d_gain, _ = linear_gain(d_diff, t)

    # B-scan — Dt[0] = deepest trace (depth[0] = 85 m)
    Dt    = d_gain[:rad_cut, :].T        # (n, rad_cut) = (n_rec, n_t)
    z_bh  = depth[:n]                   # receiver depths [m]; z_bh[0]=85 (deep) → z_bh[-1]≈0
    x_bh  = np.arange(n) * dL          # relative along-borehole positions [m]
    dt_bh = 1.0 / sf                    # time step [ns]

    limit = max(sc * np.max(np.abs(Dt)), 1.0)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(Dt, aspect='auto', cmap='seismic',
              extent=[radius[0], radius[rad_cut - 1], depth[-1], depth[0]],
              vmin=-limit, vmax=limit)
    ax.invert_yaxis()
    ax.set_xlabel('Radial distance from B2 (m)')
    ax.set_ylabel('Depth from top of B1 (m)')
    fig.savefig(OUT_DIR / 'processed' / f'prof_{run}.png', dpi=150)
    plt.close(fig)

    # --- Kirchhoff migration ---
    # z_img decreasing (85→0) so row 0 of m_kir = 85 m
    z_img   = np.linspace(z_bh.max(), z_bh.min(), n)
    recs_bh = np.vstack((np.zeros(n), z_bh))       # (x=0, z=depth) — left-wall geometry

    K = KirchhoffPylopsZeroOffset.Kirchhoff(
        z=z_img, x=x_img, t=t_mig,
        srcs=recs_bh, recs=recs_bh,
        vel=v, wav=wav_mig, wavcenter=wcenter_mig,
        mode='analytic', dynamic=False,
    )
    m_kir = (K.H @ Dt.flatten()).reshape(len(x_img), len(z_img)).T   # (n_depth, n_radial)

    lim_kir = max(sc * np.max(np.abs(m_kir)), 1.0)
    fig_k, ax_k = plt.subplots(figsize=(8, 6))
    ax_k.imshow(m_kir, aspect='auto', cmap='seismic',
                extent=[x_img[0], x_img[-1], z_img[-1], z_img[0]],
                vmin=-lim_kir, vmax=lim_kir)
    ax_k.invert_yaxis()
    ax_k.set_xlabel('Radial distance from borehole (m)')
    ax_k.set_ylabel('Depth (m)')
    fig_k.savefig(OUT_DIR / 'migrated' / f'kirchhoff_{run}.png', dpi=150)
    plt.close(fig_k)

    # --- Gazdag migration ---
    # Rotate: borehole depth → x (along-track), radial distance → z (continuation direction)
    # d_gain[:rad_cut,:] is (n_t, n_x) as required; v_mig = v/2 applied internally
    m_gaz = gazdag_migration(d_gain[:rad_cut, :], x_bh, t_mig, x_img, v)
    # m_gaz: (n_radial, n_depth) → .T gives (n_depth, n_radial); row 0 = z_bh[0] = 85 m
    lim_gaz = max(sc * np.max(np.abs(m_gaz)), 1.0)
    fig_g, ax_g = plt.subplots(figsize=(8, 6))
    ax_g.imshow(m_gaz.T, aspect='auto', cmap='seismic',
                extent=[x_img[0], x_img[-1], z_bh[-1], z_bh[0]],
                vmin=-lim_gaz, vmax=lim_gaz)
    ax_g.invert_yaxis()
    ax_g.set_xlabel('Radial distance from borehole (m)')
    ax_g.set_ylabel('Depth (m)')
    fig_g.savefig(OUT_DIR / 'migrated' / f'gazdag_{run}.png', dpi=150)
    plt.close(fig_g)

    # --- Back propagation files ---
    eps_r = (0.299792458 / v) ** 2     # permittivity from velocity (c in m/ns)
    t0_ns = 299.0 / sf                 # direct-wave end, from remove_mean window start
    write_backprop_files(
        OUT_DIR, label=f'prof_{run}', slug=f'prof_{run}',
        tapered_ntr_nt=Dt, dt_ns=dt_bh,
        x_midpoints=z_bh, t0_ns=t0_ns,
        eps_r=eps_r, v_ice=v,
        dx=bp_dx, domain_y=bp_domain_y, src_y=bp_src_y, pml_cells=bp_pml,
    )

  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=13

# Migration